# Ampère's Law — Data Exploration

The original Phyphox files are stored separately from the spreadsheets processed during the original Excel-based analysis.

- `data/raw`: original Phyphox exports
- `data/legacy_processed`: spreadsheets containing the original manual processing

In [64]:
import sys

sys.executable

'c:\\Users\\peliz\\venvs\\ampere-law-data-analysis\\Scripts\\python.exe'

In [65]:
from pathlib import Path
import pandas as pd

raw_dir = Path("../data/raw")
legacy_dir = Path("../data/legacy_processed")

raw_files = sorted(
    list(raw_dir.glob("*.xls")) +
    list(raw_dir.glob("*.xlsx"))
)

legacy_files = sorted(
    list(legacy_dir.glob("*.xls")) +
    list(legacy_dir.glob("*.xlsx"))
)

print(f"Raw files: {len(raw_files)}")
print(f"Legacy processed files: {len(legacy_files)}")

Raw files: 7
Legacy processed files: 7


In [66]:
print("Raw files:")

for file in raw_files:
    print(file.name)

Raw files:
Lei de Ampre 2026-06-25 20-11-29.xls
Lei de Ampre 2026-06-25 20-19-44.xls
Lei de Ampre 2026-06-25 20-22-32.xls
Lei de Ampre 2026-06-25 20-26-26.xls
Lei de Ampre 2026-06-25 20-28-01.xls
Lei de Ampre 2026-06-25 20-32-35.xls
Lei de Ampre 2026-06-25 20-34-16.xls


In [67]:
print("Legacy processed files:")

for file in legacy_files:
    print(file.name)

Legacy processed files:
Lei de Ampere - 0A - Dentro.xls
Lei de Ampere - 0A - Fora.xls
Lei de Ampere - 1A - Dentro.xls
Lei de Ampere - 1A - Fora.xls
Lei de Ampere - 2A - Dentro.xls
Lei de Ampere - 2A - Fora.xls
Lei de Ampere - Incerteza.xls


In [68]:
raw_excel = pd.ExcelFile(raw_files[0])

print("File:", raw_files[0].name)
print("Sheets:", raw_excel.sheet_names)

File: Lei de Ampre 2026-06-25 20-11-29.xls
Sheets: ['Gyroscope', 'Magnetometer', 'Metadata Device', 'Metadata Time']


In [69]:
metadata_raw = pd.read_excel(
    raw_files[0],
    sheet_name="Metadata Time"
)

metadata_raw

,event,experiment time,system time,system time text
0,START,0.000000,1.782429e+09,2026-06-25 20:10:50.045 UTC-03:00
1,PAUSE,34.443621,1.782429e+09,2026-06-25 20:11:24.488 UTC-03:00


In [70]:
metadata_legacy = pd.read_excel(
    legacy_files[0],
    sheet_name="Metadata Time"
)

metadata_legacy

,event,experiment time,system time,system time text
0,START,0.000000,1.782430e+09,2026-06-25 20:25:58.392 UTC-03:00
1,PAUSE,18.989061,1.782430e+09,2026-06-25 20:26:17.381 UTC-03:00


In [71]:
metadata_raw.columns

Index(['event', 'experiment time', 'system time', 'system time text'], dtype='str')

In [72]:
metadata_legacy.columns

Index(['event', 'experiment time', 'system time', 'system time text'], dtype='str')

## Matching raw and legacy files

The `Metadata Time` sheet was preserved in both the original Phyphox exports and the legacy Excel spreadsheets.

Because the raw filenames do not identify the experimental condition, the START and PAUSE timestamps are used to match each legacy spreadsheet with its original raw acquisition.

In [73]:
def get_time_signature(file):
    metadata = pd.read_excel(
        file,
        sheet_name="Metadata Time"
    )

    start = metadata.loc[
        metadata["event"] == "START",
        "system time text"
    ].iloc[0]

    pause = metadata.loc[
        metadata["event"] == "PAUSE",
        "system time text"
    ].iloc[0]

    return start, pause

In [74]:
get_time_signature(raw_files[0])

('2026-06-25 20:10:50.045 UTC-03:00', '2026-06-25 20:11:24.488 UTC-03:00')

In [75]:
get_time_signature(legacy_files[0])

('2026-06-25 20:25:58.392 UTC-03:00', '2026-06-25 20:26:17.381 UTC-03:00')

In [76]:
matches = []

for legacy_file in legacy_files:
    legacy_signature = get_time_signature(legacy_file)

    for raw_file in raw_files:
        raw_signature = get_time_signature(raw_file)

        if legacy_signature == raw_signature:
            matches.append({
                "legacy_file": legacy_file.name,
                "raw_file": raw_file.name,
                "start": legacy_signature[0],
                "pause": legacy_signature[1]
            })

matches = pd.DataFrame(matches)

matches

,legacy_file,raw_file,start,pause
0,Lei de Ampere - 0A - Dentro.xls,Lei de Ampre 2026-06-25 20-26-26.xls,2026-06-25 20:25:58.392 UTC-03:00,2026-06-25 20:26:17.381 UTC-03:00
1,Lei de Ampere - 0A - Fora.xls,Lei de Ampre 2026-06-25 20-32-35.xls,2026-06-25 20:32:10.954 UTC-03:00,2026-06-25 20:32:28.290 UTC-03:00
2,Lei de Ampere - 1A - Dentro.xls,Lei de Ampre 2026-06-25 20-19-44.xls,2026-06-25 20:18:53.261 UTC-03:00,2026-06-25 20:19:31.092 UTC-03:00
3,Lei de Ampere - 2A - Dentro.xls,Lei de Ampre 2026-06-25 20-22-32.xls,2026-06-25 20:21:48.545 UTC-03:00,2026-06-25 20:22:19.611 UTC-03:00
4,Lei de Ampere - 2A - Fora.xls,Lei de Ampre 2026-06-25 20-34-16.xls,2026-06-25 20:33:56.131 UTC-03:00,2026-06-25 20:34:12.151 UTC-03:00
5,Lei de Ampere - Incerteza.xls,Lei de Ampre 2026-06-25 20-28-01.xls,2026-06-25 20:27:42.705 UTC-03:00,2026-06-25 20:27:56.411 UTC-03:00


### Unmatched data

The metadata-based matching procedure successfully identified the original raw Phyphox acquisitions for six of the seven legacy spreadsheets.

No raw acquisition with matching `START` and `PAUSE` timestamps was found for `1A - Fora`. Therefore, this dataset was excluded from the reproducible analysis.

The remaining unmatched raw acquisition was also excluded, since its metadata could not be associated with any of the legacy experimental conditions.

The subsequent analysis uses only datasets for which the correspondence between the original raw acquisition and the documented experimental condition could be established.